In [0]:
%python
spark.sql("""
CREATE OR REPLACE TABLE data_modelling_demo.gold_snowflake.fact_sales AS
SELECT 
  o.order_id,
  o.customer_id,
  o.order_date,
  o.order_status,
  o.payment_id,
  o.order_total,
  oi.order_item_id,
  oi.product_id,
  oi.quantity,
  oi.unit_price,
  oi.discount_amount,
  oi.line_total
FROM data_modelling_demo.silver.orders o
JOIN data_modelling_demo.silver.order_items oi
  ON o.order_id = oi.order_id
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
table data_modelling_demo.gold_snowflake.fact_sales

order_id,customer_id,order_date,order_status,payment_id,order_total,order_item_id,product_id,quantity,unit_price,discount_amount,line_total
501,C001,2024-04-01,Completed,401,1079.98,602,303,1,79.99,0.0,79.99
502,C002,2024-04-02,Completed,402,899.99,603,302,1,899.99,0.0,899.99
501,C001,2024-04-01,Completed,401,1079.98,601,301,1,999.99,20.0,979.99


In [0]:
CREATE OR REPLACE TABLE data_modelling_demo.gold_snowflake.dim_geography AS
SELECT
  c.city_id,
  c.city_name,
  s.state_name,
  co.country_name
FROM data_modelling_demo.silver.cities c
JOIN data_modelling_demo.silver.states s
  ON c.state_id = s.state_id
JOIN data_modelling_demo.silver.countries co
  ON s.country_id = co.country_id

num_affected_rows,num_inserted_rows


In [0]:
table data_modelling_demo.gold_snowflake.dim_geography

city_id,city_name,state_name,country_name
1001,New York City,New York,USA
1002,San Francisco,California,USA
1003,Dallas,Texas,USA
1004,Toronto,Ontario,Canada
1005,Mumbai,Maharashtra,India


In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_customer AS
SELECT * FROM data_modelling_demo.silver.customers

In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_brand AS
SELECT * FROM data_modelling_demo.silver.brands

In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_product AS
SELECT * FROM data_modelling_demo.silver.products

In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_payment AS
SELECT * FROM data_modelling_demo.silver.payments

In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_category AS
SELECT * FROM data_modelling_demo.silver.categories

In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_snowflake.dim_sub_category AS
SELECT * FROM data_modelling_demo.silver.subcategories

In [0]:
CREATE OR REPLACE TABLE data_modelling_demo.gold_snowflake.dim_date AS
WITH date_range AS (
  SELECT EXPLODE(SEQUENCE(TO_DATE('2024-01-01'), TO_DATE('2026-12-31'), INTERVAL 1 DAY)) AS date
)
SELECT 
  date AS date_id,
  date,
  YEAR(date) AS year,
  MONTH(date) AS month,
  DAYOFMONTH(date) AS day,
  DAYOFWEEK(date) AS day_of_week,
  DAYOFYEAR(date) AS day_of_year,
  WEEKOFYEAR(date) AS week_of_year,
  QUARTER(date) AS quarter,
  DATE_FORMAT(date, 'EEEE') AS weekday_name,
  DATE_FORMAT(date, 'MMMM') AS month_name,
  CASE WHEN DAYOFWEEK(date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  -- Fiscal year assuming fiscal year starts in July (month 7)
  CASE 
    WHEN MONTH(date) >= 7 THEN YEAR(date) + 1
    ELSE YEAR(date)
  END AS fiscal_year,
  CASE 
    WHEN MONTH(date) >= 7 THEN MONTH(date) - 6
    ELSE MONTH(date) + 6
  END AS fiscal_month,
  CASE 
    WHEN MONTH(date) IN (1, 2, 3) THEN 'Q1'
    WHEN MONTH(date) IN (4, 5, 6) THEN 'Q2'
    WHEN MONTH(date) IN (7, 8, 9) THEN 'Q3'
    ELSE 'Q4'
  END AS quarter_name,
  LAST_DAY(date) AS last_day_of_month,
  CASE WHEN date = LAST_DAY(date) THEN TRUE ELSE FALSE END AS is_last_day_of_month
FROM date_range

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM data_modelling_demo.gold_snowflake.dim_date LIMIT 10

date_id,date,year,month,day,day_of_week,day_of_year,week_of_year,quarter,weekday_name,month_name,is_weekend,fiscal_year,fiscal_month,quarter_name,last_day_of_month,is_last_day_of_month
2024-01-01,2024-01-01,2024,1,1,2,1,1,1,Monday,January,false,2024,7,Q1,2024-01-31,false
2024-01-02,2024-01-02,2024,1,2,3,2,1,1,Tuesday,January,false,2024,7,Q1,2024-01-31,false
2024-01-03,2024-01-03,2024,1,3,4,3,1,1,Wednesday,January,false,2024,7,Q1,2024-01-31,false
2024-01-04,2024-01-04,2024,1,4,5,4,1,1,Thursday,January,false,2024,7,Q1,2024-01-31,false
2024-01-05,2024-01-05,2024,1,5,6,5,1,1,Friday,January,false,2024,7,Q1,2024-01-31,false
2024-01-06,2024-01-06,2024,1,6,7,6,1,1,Saturday,January,true,2024,7,Q1,2024-01-31,false
2024-01-07,2024-01-07,2024,1,7,1,7,1,1,Sunday,January,true,2024,7,Q1,2024-01-31,false
2024-01-08,2024-01-08,2024,1,8,2,8,2,1,Monday,January,false,2024,7,Q1,2024-01-31,false
2024-01-09,2024-01-09,2024,1,9,3,9,2,1,Tuesday,January,false,2024,7,Q1,2024-01-31,false
2024-01-10,2024-01-10,2024,1,10,4,10,2,1,Wednesday,January,false,2024,7,Q1,2024-01-31,false


In [0]:
SELECT 
  cat.category_name,
  d.fiscal_year,
  SUM(f.line_total) AS total_sales_amount,
  COUNT(DISTINCT f.order_id) AS total_orders,
  COUNT(f.order_item_id) AS total_items
FROM data_modelling_demo.gold_snowflake.fact_sales f
JOIN data_modelling_demo.gold_snowflake.dim_product p
  ON f.product_id = p.product_id
JOIN data_modelling_demo.gold_snowflake.dim_sub_category sc
  ON p.subcategory_id = sc.subcategory_id
JOIN data_modelling_demo.gold_snowflake.dim_category cat
  ON sc.category_id = cat.category_id
JOIN data_modelling_demo.gold_snowflake.dim_date d
  ON f.order_date = d.date
GROUP BY cat.category_name, d.fiscal_year
ORDER BY cat.category_name, d.fiscal_year

category_name,fiscal_year,total_sales_amount,total_orders,total_items
Electronics,2024,1879.98,2,2
Fashion,2024,79.99,1,1
